In [ ]:
!pip install -q git+https://github.com/huggingface/transformers.git structlog

In [ ]:
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient
login(token=UserSecretsClient().get_secret("HF_TOKEN"))

In [ ]:
!git clone -q https://github.com/Vimalk0703/Google-Gemma4-good-hackathon-VimalXMark.git /tmp/malaika-repo
import sys
sys.path.insert(0, "/tmp/malaika-repo")

import time, json, re, torch
import numpy as np
from PIL import Image

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

In [ ]:
from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig

MODEL_NAME = "google/gemma-4-E4B-it"
print(f"Loading {MODEL_NAME} in 4-bit...")
t0 = time.monotonic()

processor = AutoProcessor.from_pretrained(MODEL_NAME)
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_NAME, device_map="auto",
    quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16),
)
load_time = time.monotonic() - t0
print(f"Loaded in {load_time:.0f}s")

In [ ]:
def ask(messages, max_tokens=200):
    fmt = []
    for m in messages:
        if isinstance(m["content"], str):
            fmt.append({"role": m["role"], "content": [{"type": "text", "text": m["content"]}]})
        else:
            fmt.append(m)
    inputs = processor.apply_chat_template(
        fmt, tokenize=True, return_dict=True,
        return_tensors="pt", add_generation_prompt=True,
    ).to(model.device)
    t = time.monotonic()
    with torch.inference_mode():
        out = model.generate(**inputs, max_new_tokens=max_tokens)
    new = out[0][inputs["input_ids"].shape[1]:]
    result = processor.decode(new, skip_special_tokens=True)
    ms = (time.monotonic() - t) * 1000
    print(f"  {ms:.0f}ms | {len(new)} tokens | {len(new)/(ms/1000):.1f} tok/s")
    return result

def get_json(text):
    for pattern in [r'\{[^{}]*\}', r'```(?:json)?\s*\n?(.*?)\n?```']:
        m = re.search(pattern, text, re.DOTALL)
        if m:
            try:
                return json.loads(m.group(0) if '{' in m.group(0) else m.group(1))
            except Exception:
                pass
    return None

In [ ]:
img = Image.fromarray(np.random.randint(100, 200, (320, 240, 3), dtype=np.uint8))
img.save("/tmp/test.jpg")

from malaika.prompts import PromptRegistry
from malaika.prompts import breathing, danger_signs, diarrhea, fever, nutrition, treatment
from malaika.imci_protocol import classify_breathing, classify_danger_signs, classify_diarrhea, classify_nutrition, classify_assessment

print(f"Prompts registered: {len(PromptRegistry.list_all())}")

In [ ]:
print("TEST 1: DANGER SIGNS — Alertness")
p = PromptRegistry.get("danger.assess_alertness")
r = ask(p.render_multimodal(media={"image": "/tmp/test.jpg"}), max_tokens=300)
j = get_json(r)
print(f"  JSON: {j}")

In [ ]:
print("TEST 2: BREATHING — Chest Indrawing")
p = PromptRegistry.get("breathing.detect_chest_indrawing")
r = ask(p.render_multimodal(media={"image": "/tmp/test.jpg"}), max_tokens=300)
j = get_json(r)
print(f"  JSON: {j}")

In [ ]:
print("TEST 3: PROTOCOL — Breathing Classification")
for age, rate, exp in [(6,55,"pneumonia"), (6,42,"no_pneumonia"), (24,45,"pneumonia"), (24,35,"no_pneumonia"), (10,50,"pneumonia")]:
    c = classify_breathing(age_months=age, breathing_rate=rate, has_cough=True)
    ok = "PASS" if exp in c.classification.value else "FAIL"
    print(f"  {ok} | Age {age}mo, Rate {rate} -> {c.classification.value} ({c.severity.value})")

In [ ]:
print("TEST 4: DIARRHEA — Dehydration Signs")
p = PromptRegistry.get("diarrhea.assess_dehydration_signs")
r = ask(p.render_multimodal(media={"image": "/tmp/test.jpg"}), max_tokens=300)
j = get_json(r)
print(f"  JSON: {j}")

In [ ]:
print("TEST 5: NUTRITION — Wasting")
p = PromptRegistry.get("nutrition.assess_wasting")
r = ask(p.render_multimodal(media={"image": "/tmp/test.jpg"}), max_tokens=300)
j = get_json(r)
print(f"  JSON: {j}")

In [ ]:
print("TEST 6: TREATMENT — Swahili")
p = PromptRegistry.get("treatment.generate_plan")
r = ask(p.render(classifications="Pneumonia", urgency="YELLOW", language="Swahili", child_age_months=6), max_tokens=500)
print(f"\n{r}")

In [ ]:
print("TEST 7: GOLDEN SCENARIOS")
from malaika.evaluation.golden_scenarios import GOLDEN_SCENARIOS
passed = 0
for s in GOLDEN_SCENARIOS:
    result = classify_assessment(age_months=s.age_months, **s.findings)
    actual = set(c.value for c in result.all_classification_types)
    expected = set(c.value for c in s.expected_classifications)
    if expected.issubset(actual) and result.severity == s.expected_severity:
        passed += 1
        print(f"  PASS | {s.name}")
    else:
        print(f"  FAIL | {s.name}")
print(f"\n  Result: {passed}/{len(GOLDEN_SCENARIOS)}")

In [ ]:
print("TEST 8: JSON RELIABILITY")
prompts_to_test = ["danger.assess_alertness", "breathing.detect_chest_indrawing",
    "diarrhea.assess_dehydration_signs", "nutrition.assess_wasting", "nutrition.detect_edema"]
jp = 0
for name in prompts_to_test:
    p = PromptRegistry.get(name)
    r = ask(p.render_multimodal(media={"image": "/tmp/test.jpg"}), max_tokens=300)
    j = get_json(r)
    if j:
        jp += 1
        print(f"  PASS | {name}")
    else:
        print(f"  FAIL | {name}: {r[:80]}")
print(f"\n  JSON: {jp}/{len(prompts_to_test)}")

In [ ]:
print("=" * 50)
print("SUMMARY")
print("=" * 50)
print(f"Model:    {MODEL_NAME}")
print(f"GPU:      {torch.cuda.get_device_name(0)}")
print(f"Load:     {load_time:.0f}s")
print(f"Golden:   {passed}/{len(GOLDEN_SCENARIOS)}")
print(f"JSON:     {jp}/{len(prompts_to_test)}")
print("=" * 50)